In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set(style="whitegrid")

# -------------------------------
# Load and Preprocess the Dataset
# -------------------------------
def load_and_preprocess_data(file_path):
    """
    Loads the dataset and applies basic preprocessing.

    Parameters:
        file_path (str): Path to the dataset file (CSV format).

    Returns:
        pd.DataFrame: Preprocessed dataset.
        np.array: Scaled feature array.
    """
    # Load the dataset
    data = pd.read_csv(file_path)

    # Handle missing values (if any)
    data.fillna(method='ffill', inplace=True)

    # Encode categorical variables
    le_region = LabelEncoder()
    le_usage_type = LabelEncoder()
    le_payment_status = LabelEncoder()

    data['Region'] = le_region.fit_transform(data['Region'])
    data['Usage_Type'] = le_usage_type.fit_transform(data['Usage_Type'])
    data['Payment_Status'] = le_payment_status.fit_transform(data['Payment_Status'])

    # Extract features (customize based on your dataset)
    features = data[['Energy_Consumption_kWh', 'Region', 'Usage_Type', 'Payment_Status']]

    # Scale the features for consistency
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)

    return data, scaled_features

# ---------------------------------
# Generate Personalized Insights
# ---------------------------------
def generate_recommendations(data):
    """
    Generates customer-specific recommendations based on clustering.

    Parameters:
        data (pd.DataFrame): Dataset with clustering labels.

    Returns:
        pd.DataFrame: Dataset with recommendations added.
    """
    recommendations = {
        0: "You're in the 'Low Consumption' group. Maintain good habits!",
        1: "You're in the 'Moderate Consumption' group. Consider turning off appliances when not in use.",
        2: "You're in the 'High Consumption' group. Reduce usage during peak hours and switch to energy-efficient appliances."
    }
    
    data['Recommendation'] = data['Cluster'].map(recommendations)
    return data

# ---------------------------------
# Pattern Analysis: Clustering
# ---------------------------------
def perform_clustering(data, scaled_features, n_clusters=3):
    """
    Applies K-Means clustering to the dataset and outputs customer-friendly results.

    Parameters:
        data (pd.DataFrame): Original dataset.
        scaled_features (np.array): Scaled feature array.
        n_clusters (int): Number of clusters for K-Means.

    Returns:
        pd.DataFrame: Dataset with cluster labels and recommendations.
    """
    # Apply K-Means clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    data['Cluster'] = kmeans.fit_predict(scaled_features)

    # Generate recommendations for each cluster
    data = generate_recommendations(data)

    # Visualize clusters
    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        x=scaled_features[:, 0],
        y=scaled_features[:, 1],
        hue=data['Cluster'],
        palette='viridis'
    )
    plt.title("Clustering Analysis with Recommendations")
    plt.xlabel("Feature 1 (scaled)")
    plt.ylabel("Feature 2 (scaled)")
    plt.show()

    return data

# ---------------------------------
# Anomaly Detection
# ---------------------------------
def detect_anomalies(data, scaled_features, contamination_rate=0.05):
    """
    Detects anomalies using Isolation Forest and outputs simplified, customer-friendly results.

    Parameters:
        data (pd.DataFrame): Original dataset.
        scaled_features (np.array): Scaled feature array.
        contamination_rate (float): Proportion of anomalies in the dataset.

    Returns:
        pd.DataFrame: Dataset with anomaly labels.
    """
    # Apply Isolation Forest
    iso_forest = IsolationForest(contamination=contamination_rate, random_state=42)
    data['Anomaly'] = iso_forest.fit_predict(scaled_features)

    # Map anomaly labels to binary (1 for anomaly, 0 for normal)
    data['Anomaly'] = data['Anomaly'].apply(lambda x: 1 if x == -1 else 0)

    # Output anomaly results in a customer-friendly way
    anomaly_summary = data['Anomaly'].value_counts()
    print("\nAnomaly Detection Summary:")
    print(f"  - Normal data points: {anomaly_summary[0]}")
    print(f"  - Anomalous data points: {anomaly_summary[1]}")

    # Show a few examples of anomalies and normal data points
    print("\nExample of Anomalous Data Points (if any):")
    print(data[data['Anomaly'] == 1].head())
    print("\nExample of Normal Data Points:")
    print(data[data['Anomaly'] == 0].head())

    # Visualize anomalies
    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        x=scaled_features[:, 0],
        y=scaled_features[:, 1],
        hue=data['Anomaly'],
        palette={0: 'blue', 1: 'red'}
    )
    plt.title("Anomaly Detection")
    plt.xlabel("Feature 1 (scaled)")
    plt.ylabel("Feature 2 (scaled)")
    plt.show()

    return data

# ---------------------------------
# Predict Future Energy Consumption
# ---------------------------------
def predict_future_consumption(data):
    """
    Predicts future energy consumption using a Random Forest Regressor.

    Parameters:
        data (pd.DataFrame): Dataset containing features and target variable.

    Returns:
        None
    """
    # Define features and target variable
    features = data[['Region', 'Usage_Type', 'Payment_Status']]
    target = data['Energy_Consumption_kWh']

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

    # Train a Random Forest Regressor
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)

    # Evaluate the model
    mse = mean_squared_error(y_test, predictions)
    print(f"Mean Squared Error (MSE): {mse}")

    # Visualize actual vs predicted values
    plt.figure(figsize=(8, 6))
    plt.scatter(y_test, predictions, alpha=0.7)
    plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--', lw=2)
    plt.title("Actual vs Predicted Energy Consumption")
    plt.xlabel("Actual Energy Consumption (kWh)")
    plt.ylabel("Predicted Energy Consumption (kWh)")
    plt.show()

# ----------------------------------
# Visualize Consumption Patterns
# ----------------------------------
def visualize_consumption_patterns(data):
    """
    Visualizes energy consumption patterns as a bar graph.

    Parameters:
        data (pd.DataFrame): Dataset containing time and energy consumption columns.

    Returns:
        None
    """
    # Aggregate data by 'Region' and 'Usage_Type' (assuming regional consumption)
    region_usage_consumption = data.groupby(['Region', 'Usage_Type'])['Energy_Consumption_kWh'].sum().reset_index()

    # Plot bar graph
    plt.figure(figsize=(12, 6))
    sns.barplot(
        x='Region', y='Energy_Consumption_kWh', hue='Usage_Type', data=region_usage_consumption, palette='Set2'
    )
    plt.title("Energy Consumption by Region and Usage Type")
    plt.xlabel("Region")
    plt.ylabel("Total Energy Consumption (kWh)")
    plt.tight_layout()
    plt.show()

# ---------------------------
# Main Function for Workflow
# ---------------------------
if __name__ == "__main__":
    # File path to your dataset
    dataset_path = "energy_data.csv"  # Replace with your actual dataset path

    # Step 1: Load and preprocess data
    data, scaled_features = load_and_preprocess_data(dataset_path)

    # Step 2: Perform clustering analysis
    data = perform_clustering(data, scaled_features, n_clusters=3)

    # Step 3: Detect anomalies
    data = detect_anomalies(data, scaled_features, contamination_rate=0.05)

    # Step 4: Predict future energy consumption
    predict_future_consumption(data)

    # Step 5: Visualize energy consumption patterns
    visualize_consumption_patterns(data)

    # Step 6: Save results
    data.to_csv("processed_energy_data.csv", index=False)
    print("Results saved to 'processed_energy_data.csv'")
